# MLflow Pipeline Evidence

Concrete evidence of the many-model forecasting pipeline execution:
- **17 models** trained and evaluated across **99 time series** with **10 backtest windows**
- **16,830 evaluation records** logged to Unity Catalog
- **1,683 production forecasts** generated in scoring output
- All artifacts tracked via MLflow run ID linked to evaluation/scoring tables

In [0]:
eval_results = spark.sql("""
    SELECT model, run_id,
        COUNT(DISTINCT unique_id) as series_evaluated,
        COUNT(DISTINCT backtest_window_start_date) as backtest_windows,
        COUNT(*) as total_evaluations,
        ROUND(AVG(metric_value), 6) as avg_smape,
        ROUND(MIN(metric_value), 6) as best_smape,
        ROUND(MAX(metric_value), 6) as worst_smape
    FROM mmf_demo_aym.m4_advanced.monthly_evaluation_output
    GROUP BY model, run_id ORDER BY avg_smape ASC
""")
display(eval_results)

Many-Model Forecasting Pipeline — Evaluation Results
Run ID: 83517d0d-584f-43f6-b505-a442fccdf763
Total evaluations: 16,830
Models evaluated: 17

Model                                     Series  Windows   Evals   Avg sMAPE      Best     Worst
----------------------------------------------------------------------------------------------------
StatsForecastBaselineSeasonalWindowAverage      99       10     990         NaN       NaN       NaN
StatsForecastAutoCES                          99       10     990    0.061535  0.000436  0.525880
StatsForecastAutoETS                          99       10     990    0.062436  0.000218  0.497261
StatsForecastAutoArima                        99       10     990    0.063498  0.000337  0.402578
SKTimeProphet                                 99       10     990    0.066887  0.000100  0.413974
StatsForecastAutoTheta                        99       10     990    0.068614  0.000172  0.585924
StatsForecastBaselineSeasonalNaive            99       10     990

In [0]:
scoring_results = spark.sql("""
    SELECT model, COUNT(DISTINCT unique_id) as series_scored, run_id
    FROM mmf_demo_aym.m4_advanced.monthly_scoring_output
    GROUP BY model, run_id ORDER BY model
""")
display(scoring_results)

Scoring Output — Production Forecasts
Run ID: 83517d0d-584f-43f6-b505-a442fccdf763
Model                                     Series Scored
--------------------------------------------------------
SKTimeProphet                                        99
StatsForecastADIDA                                   99
StatsForecastAutoArima                               99
StatsForecastAutoCES                                 99
StatsForecastAutoETS                                 99
StatsForecastAutoMfles                               99
StatsForecastAutoTbats                               99
StatsForecastAutoTheta                               99
StatsForecastBaselineNaive                           99
StatsForecastBaselineSeasonalNaive                   99
StatsForecastBaselineSeasonalWindowAverage             99
StatsForecastBaselineWindowAverage                   99
StatsForecastCrostonClassic                          99
StatsForecastCrostonOptimized                        99
StatsForecastCrost

In [0]:
best_models = spark.sql("""
    WITH ranked AS (
        SELECT unique_id, model, AVG(metric_value) as avg_smape,
            ROW_NUMBER() OVER (PARTITION BY unique_id ORDER BY AVG(metric_value)) as rn
        FROM mmf_demo_aym.m4_advanced.monthly_evaluation_output
        WHERE metric_name = 'smape'
        GROUP BY unique_id, model
    )
    SELECT unique_id, model as best_model, ROUND(avg_smape, 6) as avg_smape
    FROM ranked WHERE rn = 1
    ORDER BY unique_id LIMIT 10
""")
display(best_models)

unique_id    best_model                           avg_smape
------------------------------------------------------------
Error: unsupported format string passed to NoneType.__format__
